# Customer Churn Prediction Using Logistic Regression

## Project overview
This project predicts whether a customer will churn using subscription length, complaint history, satisfaction score, and discount usage.

The notebook demonstrates an end-to-end classification workflow:
- data validation and exploratory analysis
- train/test splitting
- logistic regression modeling
- evaluation using precision, recall, F1-score, confusion matrix, and ROC-AUC
- coefficient and odds-ratio interpretation
- business recommendations

> **Verified result from the original run:** 86% test accuracy and 0.861 ROC-AUC.


## Business problem

Customer churn can reduce recurring revenue and increase acquisition costs. The objective is to identify customers at risk of leaving and understand which observable factors are most associated with churn.

### Target
- `Churn = 1`: Customer churned
- `Churn = 0`: Customer did not churn

### Predictors
- `SubscriptionLength`
- `ComplaintsFiled`
- `SatisfactionScore`
- `DiscountUsed`


In [ ]:
# Core libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
)

pd.set_option("display.max_columns", None)


## 1. Load the dataset

Place `customer_sales_churn.csv` in the `data/` folder when running locally. In Google Colab, the notebook will prompt you to upload the file when it is not already available.


In [ ]:
from pathlib import Path

local_paths = [
    Path("data/customer_sales_churn.csv"),
    Path("customer_sales_churn.csv"),
]

data_path = next((path for path in local_paths if path.exists()), None)

if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        data_path = Path(uploaded_name)
    except Exception as exc:
        raise FileNotFoundError(
            "Dataset not found. Add customer_sales_churn.csv to the data/ folder "
            "or upload it when running in Google Colab."
        ) from exc

df = pd.read_csv(data_path)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")
df.head()


## 2. Data quality checks


In [ ]:
required_columns = [
    "SubscriptionLength",
    "ComplaintsFiled",
    "SatisfactionScore",
    "DiscountUsed",
    "Churn",
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(),
})

display(quality_summary)
print(f"Duplicate rows: {df.duplicated().sum()}")


In [ ]:
df[required_columns].describe().T


## 3. Exploratory data analysis


In [ ]:
target_counts = df["Churn"].value_counts().sort_index()
target_rates = df["Churn"].value_counts(normalize=True).sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", ax=ax)
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Churn status")
ax.set_ylabel("Number of customers")
ax.set_xticklabels(["No churn (0)", "Churn (1)"], rotation=0)

for index, count in enumerate(target_counts):
    ax.text(index, count, f"{count}\n({target_rates.iloc[index]:.1%})",
            ha="center", va="bottom")

plt.tight_layout()
plt.show()


In [ ]:
features = [
    "SubscriptionLength",
    "ComplaintsFiled",
    "SatisfactionScore",
    "DiscountUsed",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for feature, ax in zip(features, axes.flatten()):
    df.boxplot(column=feature, by="Churn", ax=ax)
    ax.set_title(f"{feature} by Churn")
    ax.set_xlabel("Churn")
    ax.set_ylabel(feature)

plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
correlation = df[features + ["Churn"]].corr()

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(correlation, aspect="auto")
ax.set_xticks(range(len(correlation.columns)))
ax.set_yticks(range(len(correlation.index)))
ax.set_xticklabels(correlation.columns, rotation=45, ha="right")
ax.set_yticklabels(correlation.index)

for row in range(len(correlation.index)):
    for col in range(len(correlation.columns)):
        ax.text(col, row, f"{correlation.iloc[row, col]:.2f}",
                ha="center", va="center")

fig.colorbar(image, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()


## 4. Prepare training and test data

A stratified split keeps the churn proportion similar in the training and test sets.


In [ ]:
X = df[features].copy()
y = df["Churn"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Training churn rate:", f"{y_train.mean():.1%}")
print("Test churn rate:", f"{y_test.mean():.1%}")


## 5. Train the logistic regression model


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


## 6. Evaluate model performance


In [ ]:
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob),
    ],
})

metrics["Score"] = metrics["Score"].round(3)
metrics


In [ ]:
print(classification_report(y_test, y_pred, digits=3))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No churn", "Churn"],
    ax=ax,
    values_format="d",
)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("ROC Curve")
plt.tight_layout()
plt.show()


### Verified findings from the original notebook run

The original 70/30 split produced:

| Metric | Result |
|---|---:|
| Accuracy | 0.86 |
| ROC-AUC | 0.861 |
| Churn precision | 0.66 |
| Churn recall | 0.46 |
| Churn F1-score | 0.54 |

The model distinguished churners from non-churners reasonably well, but it missed more than half of actual churners. For a retention program, recall is especially important because a false negative represents a customer at risk who receives no intervention.

> The cleaned notebook uses stratification, so results may differ slightly when it is rerun.


## 7. Interpret coefficients and odds ratios


In [ ]:
coefficient_table = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0],
    "Odds_Ratio": np.exp(model.coef_[0]),
}).sort_values("Coefficient", ascending=False)

coefficient_table.round(3)


In [ ]:
plot_table = coefficient_table.sort_values("Coefficient")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_table["Feature"], plot_table["Coefficient"])
ax.axvline(0, linewidth=1)
ax.set_title("Logistic Regression Coefficients")
ax.set_xlabel("Coefficient")
plt.tight_layout()
plt.show()


### Interpretation of the original coefficients

The original model produced approximately:

| Feature | Coefficient | Interpretation |
|---|---:|---|
| ComplaintsFiled | +0.867 | More complaints were associated with higher churn odds. |
| DiscountUsed | +0.752 | Discount use was associated with higher churn odds in this dataset. This is association, not proof that discounts cause churn. |
| SubscriptionLength | -0.059 | Longer subscriptions were associated with lower churn odds. |
| SatisfactionScore | -0.324 | Higher satisfaction was associated with lower churn odds. |

For a logistic regression coefficient, `exp(coefficient)` gives the multiplicative change in the odds for a one-unit increase, holding the other variables constant.


## 8. Optional threshold analysis

The default classification threshold is 0.50. A lower threshold can improve churn recall, but it usually increases false positives. The table below helps select a threshold based on business priorities.


In [ ]:
threshold_rows = []

for threshold in np.arange(0.20, 0.81, 0.05):
    threshold_pred = (y_prob >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(y_test, threshold_pred, zero_division=0),
        "Recall": recall_score(y_test, threshold_pred, zero_division=0),
        "F1-score": f1_score(y_test, threshold_pred, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_rows)
threshold_results.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_results["Threshold"], threshold_results["Precision"], marker="o", label="Precision")
ax.plot(threshold_results["Threshold"], threshold_results["Recall"], marker="o", label="Recall")
ax.plot(threshold_results["Threshold"], threshold_results["F1-score"], marker="o", label="F1-score")
ax.set_title("Precision and Recall by Classification Threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Business recommendations

1. **Prioritize complaint resolution.** Complaint count had the strongest positive coefficient in the original model. Create alerts for repeated complaints and assign high-risk cases to a retention team.

2. **Strengthen customer satisfaction programs.** Higher satisfaction was associated with lower churn. Track satisfaction after support interactions and follow up on low scores.

3. **Encourage longer-term relationships.** Longer subscription length was associated with lower churn. Loyalty rewards or renewal incentives may help, but they should be tested through controlled experiments.

4. **Review discount strategy carefully.** Discount use was positively associated with churn in the model. This may indicate that discounts are being offered to already at-risk customers, so the relationship should not be interpreted as causal.

5. **Optimize for recall when intervention costs are manageable.** The original model's churn recall was 46%. Lowering the probability threshold may identify more churners at the cost of contacting more customers who would not have churned.


## 10. Limitations and next steps

- The model uses only four predictors and may omit important customer behavior.
- Logistic regression captures linear relationships in log-odds.
- Coefficients show association, not causation.
- The churn class is less common, so accuracy alone can be misleading.
- Future work could compare logistic regression with tree-based models, use cross-validation, tune the classification threshold, and validate performance on newer customer cohorts.


## Conclusion

The logistic regression model provides an interpretable baseline for customer churn prediction. Its ROC-AUC indicates useful ranking ability, while the lower churn recall highlights an opportunity to adjust the decision threshold and incorporate additional predictors. The strongest actionable signal in the original model was complaint history.
